In [1]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')


file_path = '../../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

base_learners = [
    ('catboost', CatBoostClassifier(random_state=42, verbose=False)),
    ('logreg', LogisticRegression(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42)),
    ('extra_trees', ExtraTreesClassifier(random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))
]

meta_model = XGBClassifier(eval_metric='logloss')

stacking_model = StackingClassifier(estimators=base_learners, final_estimator=meta_model)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    stacking_model.fit(X_train, y_train)
    
    y_pred = stacking_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    accuracies.append(acc)

print(f'Mean Accuracy: {sum(accuracies) / len(accuracies):.4f}')

Mean Accuracy: 0.9935
